# 🔮 Customer Churn Prediction — Machine Learning Pipeline

**Goal:** Predict whether a customer is **Active** (made at least one order in the 365 days before the reference date `2022-06-08`) or **Not Active** (no order in that window). This matches the `is_active` label produced by `gold.vw_customer_activity`.

| Label | Meaning |
|-------|----------|
| **1** | ACTIVE — ordered within the 365 days before the reference date |
| **0** | NOT ACTIVE — no orders in the 365 days before the reference date |

**Data Source:** Existing Gold Layer in PostgreSQL (`gold` schema).

---

## 0. Setup & Imports

In [6]:
# ── Standard Libraries ──
import sys, os, warnings
warnings.filterwarnings('ignore')

# ── Data ──
import numpy as np
import pandas as pd

# ── Visualization ──
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# ── Scikit-learn ──
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    classification_report
)

# ── XGBoost ──
from xgboost import XGBClassifier

# ── Database (reusing existing project connection) ──
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))
from depi_project.dbconnection import get_engine

print('✅ All imports loaded successfully.')

✅ All imports loaded successfully.


---
## 1. Data Extraction from Gold Layer

We connect to the **existing PostgreSQL database** using the project's `get_engine()` function (reads credentials from `.env`).

We'll query the Gold Layer tables directly:
- **`gold.fact_sales`** — every order-item transaction with dates
- **`gold.dim_user`** — customer demographics
- **`gold.dim_date`** — date dimension to convert date keys to real dates
- **`gold.dim_product`** — product information
- **`gold.vw_customer_activity`** — pre-built view with first/last order & is_active flag
> The extraction query is split into **single-grain CTEs** (order level, product mix) and joined **1:1 on `user_key`**. **Every feature is computed only from orders before `2021-06-08`** (the start of the label window), so features cannot see the target period.


In [7]:
# ── Connect to the existing database ──
engine = get_engine()
print(f'✅ Connected to: {engine.url.database} @ {engine.url.host}')

✅ Connected to: depi_db @ 100.79.25.65


### 1.1 Inspect Gold Layer Tables

In [8]:
# ── List all tables and views in the gold schema ──
gold_objects = pd.read_sql("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'gold'
    ORDER BY table_type, table_name
""", engine)

print('📋 Gold Layer Objects:')
display(gold_objects)

📋 Gold Layer Objects:


,table_name,table_type
0,dim_date,BASE TABLE
1,dim_distribution_center,BASE TABLE
2,dim_event_profile,BASE TABLE
3,dim_location,BASE TABLE
4,dim_product,BASE TABLE
5,dim_user,BASE TABLE
6,fact_events,BASE TABLE
7,fact_inventory,BASE TABLE
8,fact_sales,BASE TABLE
9,vw_customer_activity,VIEW


In [9]:
# ── Inspect columns of key tables ──
for tbl in ['dim_user', 'dim_date', 'dim_product', 'fact_sales', 'vw_customer_activity']:
    cols = pd.read_sql(f"""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'gold' AND table_name = '{tbl}'
        ORDER BY ordinal_position
    """, engine)
    print(f'\n📌 gold.{tbl}  ({len(cols)} columns):')
    display(cols)


📌 gold.dim_user  (9 columns):


,column_name,data_type
0,user_key,integer
1,user_id,text
2,full_name,character varying
3,age,smallint
4,age_group,character varying
5,gender,character varying
6,acquisition_source,character varying
7,created_date_key,integer
8,location_key,integer



📌 gold.dim_date  (8 columns):


,column_name,data_type
0,date_key,integer
1,full_date,date
2,day,smallint
3,month,smallint
4,month_name,character varying
5,quarter,smallint
6,year,smallint
7,weekday,character varying



📌 gold.dim_product  (8 columns):


,column_name,data_type
0,product_key,integer
1,product_id,text
2,name,text
3,brand,character varying
4,category,character varying
5,department,character varying
6,retail_price,numeric
7,cost,numeric



📌 gold.fact_sales  (13 columns):


,column_name,data_type
0,order_item_id,text
1,order_id,text
2,inventory_item_id,text
3,user_key,integer
4,product_key,integer
5,order_date_key,integer
6,shipped_date_key,integer
7,delivered_date_key,integer
8,returned_date_key,integer
9,status,character varying



📌 gold.vw_customer_activity  (5 columns):


,column_name,data_type
0,user_id,text
1,first_order_date,date
2,last_order_date,date
3,days_since_last_order,integer
4,is_active,integer


### 1.2 Extract Customer-Level ML Dataset

We build a **single SQL query** that joins Gold Layer tables to produce **one row per customer**:

- **Target:** `is_active` from `gold.vw_customer_activity`
  (1 = at least one order in the 365 days before `2022-06-08`, 0 = none)
- **Features:** aggregated from `gold.fact_sales` + `gold.dim_user` + `gold.dim_date` + `gold.dim_product`, using **only orders before `2021-06-08`**

**Query structure — every CTE has one grain, and all joins to the customer grain are 1:1:**

1. `customer_orders` — one row per customer/order (pre-cutoff orders only).
2. `customer_product_mix` — distinct categories/departments per customer.
3. `customer_agg` — customer-level roll-up of `customer_orders` only (including `recency` at the cutoff and recent-window counts via `FILTER`).
4. Final `SELECT` — 1:1 joins to `dim_user`, product mix, and the activity view.

> ⚠️ **Leakage guard:** all features stop at `FEATURE_CUTOFF = 2021-06-08`. Customers whose first order is *after* the cutoff have no history to build features from, so they are not in the dataset (this is expected: we predict future activity for customers that already existed at the cutoff).


In [ ]:
# ── Build customer-level dataset from Gold Layer ──
# Reference date used by gold.vw_customer_activity and its is_active label
REFERENCE_DATE = '2022-06-08'
LABEL_WINDOW_DAYS = 365
# Label window = [FEATURE_CUTOFF, REFERENCE_DATE). ALL features are built only from
# orders strictly BEFORE FEATURE_CUTOFF, so nothing in X can see the label window.
FEATURE_CUTOFF = (pd.Timestamp(REFERENCE_DATE) - pd.Timedelta(days=LABEL_WINDOW_DAYS)).date().isoformat()
print(f'📅 Reference date: {REFERENCE_DATE} | feature cutoff: {FEATURE_CUTOFF}')

query = f"""
WITH customer_orders AS (
    -- Grain: one row per customer/order, using ONLY orders before the label window.
    SELECT
        fs.user_key,
        fs.order_id,
        dd.full_date                       AS order_date,
        SUM(fs.sale_price)                 AS order_revenue,
        SUM(fs.profit)                     AS order_profit,
        COUNT(*)                           AS items_in_order
    FROM gold.fact_sales fs
    JOIN gold.dim_date dd ON dd.date_key = fs.order_date_key
    WHERE fs.user_key <> -1                -- exclude Guest
      AND dd.full_date < DATE '{FEATURE_CUTOFF}'
    GROUP BY fs.user_key, fs.order_id, dd.full_date
),
customer_product_mix AS (
    -- Grain: one row per customer. Only COUNT(DISTINCT ...) measures live here.
    SELECT
        fs.user_key,
        COUNT(DISTINCT dp.category)         AS distinct_categories,
        COUNT(DISTINCT dp.department)       AS distinct_departments
    FROM gold.fact_sales fs
    JOIN gold.dim_product dp ON dp.product_key = fs.product_key
    JOIN gold.dim_date dd ON dd.date_key = fs.order_date_key
    WHERE fs.user_key <> -1
      AND dd.full_date < DATE '{FEATURE_CUTOFF}'
    GROUP BY fs.user_key
),
customer_agg AS (
    -- Grain: one row per customer, built ONLY from the order-grain CTE.
    SELECT
        co.user_key,
        COUNT(DISTINCT co.order_id)         AS total_orders,
        SUM(co.items_in_order)              AS total_items,
        SUM(co.order_revenue)               AS total_revenue,
        SUM(co.order_profit)                AS total_profit,
        SUM(co.order_revenue)::FLOAT
            / COUNT(DISTINCT co.order_id)   AS avg_order_value,
        SUM(co.items_in_order)::FLOAT
            / COUNT(DISTINCT co.order_id)   AS avg_items_per_order,
        -- Recency measured at the cutoff (last order is always < cutoff here)
        DATE '{FEATURE_CUTOFF}' - MAX(co.order_date) AS recency,
        DATE '{FEATURE_CUTOFF}' - MIN(co.order_date) AS tenure_days,
        MAX(co.order_date) - MIN(co.order_date) AS customer_lifetime_days,
        CASE
            WHEN COUNT(DISTINCT co.order_id) > 1
            THEN (MAX(co.order_date) - MIN(co.order_date))::FLOAT
                 / (COUNT(DISTINCT co.order_id) - 1)
            ELSE 0
        END                                 AS avg_days_between_orders,
        CASE
            WHEN MAX(co.order_date) - MIN(co.order_date) > 0
            THEN COUNT(DISTINCT co.order_id)::FLOAT
                 / ((MAX(co.order_date) - MIN(co.order_date))::FLOAT / 30.0)
            ELSE COUNT(DISTINCT co.order_id)::FLOAT
        END                                 AS purchase_frequency,
        -- Recent-window behaviour, all measured BEFORE the cutoff
        COUNT(DISTINCT co.order_id) FILTER (
            WHERE co.order_date >= DATE '{FEATURE_CUTOFF}' - 90)   AS orders_last_90d,
        COUNT(DISTINCT co.order_id) FILTER (
            WHERE co.order_date >= DATE '{FEATURE_CUTOFF}' - 180)  AS orders_last_180d,
        COUNT(DISTINCT co.order_id) FILTER (
            WHERE co.order_date >= DATE '{FEATURE_CUTOFF}' - 360
              AND co.order_date <  DATE '{FEATURE_CUTOFF}' - 180)  AS orders_prev_180d,
        COALESCE(SUM(co.order_revenue) FILTER (
            WHERE co.order_date >= DATE '{FEATURE_CUTOFF}' - 180), 0) AS revenue_last_180d
    FROM customer_orders co
    GROUP BY co.user_key
)
SELECT
    du.user_id,
    du.age,
    du.gender,
    du.acquisition_source,
    ca.total_orders,
    ca.total_items,
    ca.total_revenue,
    ca.total_profit,
    ca.avg_order_value,
    ca.avg_items_per_order,
    ca.recency,
    ca.customer_lifetime_days,
    ca.avg_days_between_orders,
    ca.purchase_frequency,
    ca.tenure_days,
    ca.orders_last_90d,
    ca.orders_last_180d,
    ca.orders_prev_180d,
    ca.revenue_last_180d,
    cpm.distinct_categories,
    cpm.distinct_departments,
    -- Target variable from existing view (orders inside the label window)
    vca.is_active
FROM customer_agg ca
JOIN customer_product_mix cpm
    ON cpm.user_key = ca.user_key                       -- 1:1
JOIN gold.dim_user du
    ON du.user_key = ca.user_key                        -- 1:1 (user_key is PK)
JOIN gold.vw_customer_activity vca
    ON vca.user_id = du.user_id                         -- 1:1 (one view row per user_id)
WHERE du.user_key <> -1;
"""

df = pd.read_sql(query, engine)
df['is_active'] = df['is_active'].astype(int)   # make sure the target is 0/1 ints
print(f'✅ Extracted {len(df):,} customers from Gold Layer')
print(f'   Columns: {list(df.columns)}')
df.head()


### 1.3 Extraction Validation

The extraction is checked before any modeling: **schema**, **one row per customer**, **nulls**,
**aggregation values against an independent item-level query**, **target integrity**, and
**no deterministic leakage** between `recency` and `is_active`.


In [ ]:
# ── Validate customer-level extraction ──
EXPECTED_COLUMNS = [
    'user_id', 'age', 'gender', 'acquisition_source',
    'total_orders', 'total_items', 'total_revenue', 'total_profit',
    'avg_order_value', 'avg_items_per_order', 'recency',
    'customer_lifetime_days', 'avg_days_between_orders', 'purchase_frequency',
    'tenure_days', 'orders_last_90d', 'orders_last_180d', 'orders_prev_180d', 'revenue_last_180d',
    'distinct_categories', 'distinct_departments', 'is_active'
]
assert list(df.columns) == EXPECTED_COLUMNS, 'Output schema changed'
assert df['user_id'].is_unique, 'Duplicate customers in extraction'
assert df.isnull().sum().sum() == 0, 'Nulls found in extracted dataset'
print(f'✅ Schema: {len(df.columns)} expected columns, {len(df):,} unique customers, 0 nulls')

# Independent item-grain cross-check (does not reuse the extraction query's CTEs).
truth = pd.read_sql(f"""
    SELECT du.user_id,
           COUNT(DISTINCT fs.order_id) AS total_orders,
           COUNT(*) AS total_items,
           SUM(fs.sale_price) AS total_revenue,
           SUM(fs.profit) AS total_profit,
           SUM(fs.sale_price)::FLOAT / COUNT(DISTINCT fs.order_id) AS avg_order_value,
           COUNT(*)::FLOAT / COUNT(DISTINCT fs.order_id) AS avg_items_per_order,
           COUNT(DISTINCT dp.category) AS distinct_categories,
           COUNT(DISTINCT dp.department) AS distinct_departments,
           DATE '{FEATURE_CUTOFF}' - MIN(dd.full_date) AS tenure_days,
           COUNT(DISTINCT fs.order_id) FILTER (WHERE dd.full_date >= DATE '{FEATURE_CUTOFF}' - 90)  AS orders_last_90d,
           COUNT(DISTINCT fs.order_id) FILTER (WHERE dd.full_date >= DATE '{FEATURE_CUTOFF}' - 180) AS orders_last_180d,
           COUNT(DISTINCT fs.order_id) FILTER (WHERE dd.full_date >= DATE '{FEATURE_CUTOFF}' - 360
                                                 AND dd.full_date <  DATE '{FEATURE_CUTOFF}' - 180) AS orders_prev_180d,
           COALESCE(SUM(fs.sale_price) FILTER (WHERE dd.full_date >= DATE '{FEATURE_CUTOFF}' - 180), 0) AS revenue_last_180d,
           MAX(vca.is_active) AS is_active_truth
    FROM gold.fact_sales fs
    JOIN gold.dim_date dd ON dd.date_key = fs.order_date_key
    JOIN gold.dim_user du ON du.user_key = fs.user_key
    JOIN gold.dim_product dp ON dp.product_key = fs.product_key
    JOIN gold.vw_customer_activity vca ON vca.user_id = du.user_id
    WHERE fs.user_key <> -1
      AND fs.order_date_key <= (SELECT MAX(date_key) FROM gold.dim_date WHERE full_date < DATE '{FEATURE_CUTOFF}')
    GROUP BY du.user_id
""", engine)

check = df.merge(truth, on='user_id', validate='one_to_one', suffixes=('', '_truth'))
metric_cols = [
    'total_orders', 'total_items', 'total_revenue', 'total_profit',
    'avg_order_value', 'avg_items_per_order', 'distinct_categories', 'distinct_departments',
    'tenure_days', 'orders_last_90d', 'orders_last_180d', 'orders_prev_180d', 'revenue_last_180d'
]
max_diff = {c: float((check[c].astype(float) - check[c + '_truth'].astype(float)).abs().max())
            for c in metric_cols}
print('✅ Max abs difference vs independent item-grain aggregation:')
for c, d in max_diff.items():
    print(f'   {c:24s} {d:.8f}')
assert all(d < 1e-6 for d in max_diff.values()), 'Aggregation mismatch / JOIN fan-out'

# Target integrity: values come from the view and match it for every customer.
assert set(df['is_active'].unique()) <= {0, 1}
assert (check['is_active'] == check['is_active_truth']).all(), 'Target does not match gold view'
assert not {'first_order_date', 'last_order_date', 'days_since_last_order'} & set(df.columns)
print(f'✅ Target matches gold.vw_customer_activity for all {len(check):,} customers')

# Leakage guard: every feature comes from orders before the label window,
# so no single feature should be a (near-)deterministic proxy for the target.
feature_cols = [c for c in df.columns if c not in ('user_id', 'is_active')]
num_corr = df[feature_cols].select_dtypes('number').corrwith(df['is_active']).abs()
print('✅ Leakage check: max |corr(feature, is_active)| =',
      f'{num_corr.max():.4f} ({num_corr.idxmax()})')
assert num_corr.max() < 0.95, 'A feature is still a near-deterministic proxy for the target'
print(f"   Class balance after cutoff filter: {df['is_active'].mean():.2%} active")


---
## 2. Target Definition Verification

Let's verify the target follows the exact specification:
- **1 = ACTIVE:** Customer made at least one order in the 365 days **before** `2022-06-08`
- **0 = NOT ACTIVE:** Customer made no orders in those 365 days
> This is the definition implemented by `gold.vw_customer_activity`; the label is preserved as extracted.


In [ ]:
# ── Verify target labels ──
print('📊 Target Distribution (is_active):')
print(df['is_active'].value_counts())
print()
print('📊 Target Proportions:')
print(df['is_active'].value_counts(normalize=True).round(4))
print()
print('Label Mapping:')
print('  1 = ACTIVE      → customer ordered within 365 days before', REFERENCE_DATE)
print('  0 = NOT ACTIVE   → customer did NOT order within 365 days before', REFERENCE_DATE)

---
## 3. Feature Engineering Summary

Here's a summary of the customer-level features we extracted:

| Feature | Description | Type |
|---------|-------------|------|
| `age` | Customer's age | Numerical |
| `gender` | Customer's gender | Categorical |
| `acquisition_source` | How the customer was acquired (traffic source) | Categorical |
| `total_orders` | Total number of distinct orders | Numerical |
| `total_items` | Total number of items purchased | Numerical |
| `total_revenue` | Sum of all sale prices | Numerical |
| `total_profit` | Sum of all profits | Numerical |
| `avg_order_value` | Average revenue per order | Numerical |
| `avg_items_per_order` | Average number of items per order | Numerical |
| `recency` | Days between last order and `2021-06-08` | Numerical |
| `customer_lifetime_days` | Days between first and last order | Numerical |
| `avg_days_between_orders` | Average gap between consecutive orders | Numerical |
| `purchase_frequency` | Orders per 30-day period | Numerical |
| `distinct_categories` | Number of unique product categories purchased | Numerical |
| `distinct_departments` | Number of unique product departments purchased | Numerical |
| `tenure_days` | Days between first order and `2021-06-08` | Numerical |
| `orders_last_90d` / `orders_last_180d` | Orders in the 90 / 180 days before the cutoff | Numerical |
| `orders_prev_180d` | Orders in the 180 days before that (days 180–360 before cutoff) | Numerical |
| `revenue_last_180d` | Revenue in the 180 days before the cutoff | Numerical |
> All features use orders **before 2021-06-08** only; the target `is_active` describes the following 365 days.


In [ ]:
# ── Quick overview of the dataset ──
print(f'Dataset Shape: {df.shape}')
print(f'\n📋 Data Types:')
print(df.dtypes)
print(f'\n📋 Basic Statistics:')
df.describe().round(2)

---
## 4. Data Preparation

### 4.1 Check Missing Values

In [ ]:
# ── Missing values check ──
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

if missing_df.empty:
    print('✅ No missing values found!')
else:
    print('⚠️ Missing values detected:')
    display(missing_df)

### 4.2 Check Duplicates

In [ ]:
# ── Duplicate check ──
dup_count = df.duplicated(subset=['user_id']).sum()
print(f'Duplicate user_id entries: {dup_count}')

full_dup = df.duplicated().sum()
print(f'Fully duplicated rows: {full_dup}')

if dup_count > 0:
    print('⚠️ Removing duplicates based on user_id (keeping first)...')
    df = df.drop_duplicates(subset=['user_id'], keep='first').reset_index(drop=True)
    print(f'   After dedup: {len(df):,} rows')
else:
    print('✅ No duplicates found!')

### 4.3 Handle Outliers

We'll use the **IQR method** to detect outliers and **cap** them (winsorize) rather than remove them, since extreme spenders may still be valid customers.

In [ ]:
# ── Define numerical feature columns ──
numerical_features = [
    'age', 'total_orders', 'total_items', 'total_revenue', 'total_profit',
    'avg_order_value', 'avg_items_per_order', 'recency',
    'customer_lifetime_days', 'avg_days_between_orders',
    'purchase_frequency', 'distinct_categories', 'distinct_departments',
    'tenure_days', 'orders_last_90d', 'orders_last_180d', 'orders_prev_180d', 'revenue_last_180d'
]

categorical_features = ['gender', 'acquisition_source']

print(f'Numerical features ({len(numerical_features)}): {numerical_features}')
print(f'Categorical features ({len(categorical_features)}): {categorical_features}')

In [ ]:
# ── Detect outliers with IQR ──
print('📊 Outlier Detection (IQR Method):\n')
outlier_cols = ['total_orders', 'total_items', 'total_revenue', 'total_profit',
                'avg_order_value', 'avg_items_per_order',
                'avg_days_between_orders', 'purchase_frequency',
                'orders_last_90d', 'orders_last_180d', 'orders_prev_180d', 'revenue_last_180d']

for col in outlier_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    pct = outliers / len(df) * 100
    print(f'  {col:30s} → {outliers:5d} outliers ({pct:.1f}%)  [range: {lower:.1f} – {upper:.1f}]')

In [ ]:
# ── Cap outliers (winsorization) ──
# We cap at the 1st and 99th percentiles to preserve data while limiting extreme values.
df_capped = df.copy()

for col in outlier_cols:
    p01 = df_capped[col].quantile(0.01)
    p99 = df_capped[col].quantile(0.99)
    before = len(df_capped[(df_capped[col] < p01) | (df_capped[col] > p99)])
    df_capped[col] = df_capped[col].clip(lower=p01, upper=p99)
    print(f'  {col:30s} → capped {before:5d} values to [{p01:.2f}, {p99:.2f}]')

df = df_capped
print('\n✅ Outliers capped successfully.')

### 4.4 Encode Categorical Features

In [ ]:
# ── Inspect categorical values ──
for col in categorical_features:
    print(f'\n{col} — unique values ({df[col].nunique()}):')
    print(df[col].value_counts())

In [ ]:
# ── One-Hot Encode categorical features ──
# We use pd.get_dummies with drop_first=True to avoid multicollinearity.
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True, dtype=int)

print(f'Shape after encoding: {df_encoded.shape}')
print(f'New columns: {[c for c in df_encoded.columns if c not in df.columns]}')
df_encoded.head()

### 4.5 Prepare Features and Target (Avoiding Data Leakage)

**Important:** We drop `user_id` (identifier, not a feature) and `is_active` (our target). Section 1.3 already verified that `recency` is not a deterministic proxy for the target and that view columns `last_order_date` / `days_since_last_order` were never extracted as features.

In [ ]:
# ── Separate features (X) and target (y) ──
drop_cols = ['user_id', 'is_active']
X = df_encoded.drop(columns=drop_cols)
y = df_encoded['is_active']

print(f'Features shape: {X.shape}')
print(f'Target shape:   {y.shape}')
print(f'\nFeature columns ({len(X.columns)}):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')
print(f'\nTarget distribution:')
print(y.value_counts())

In [ ]:
# ── Train / Test split (80/20, stratified to maintain class balance) ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ensures both sets have the same Active/Not Active ratio
)

print(f'Training set: {X_train.shape[0]:,} samples')
print(f'Test set:     {X_test.shape[0]:,} samples')
print(f'\nTraining target distribution:')
print(y_train.value_counts(normalize=True).round(4))
print(f'\nTest target distribution:')
print(y_test.value_counts(normalize=True).round(4))

In [ ]:
# ── Scale numerical features ──
# We fit the scaler ONLY on training data to avoid data leakage,
# then transform both training and test data.

# Identify which columns in X are numerical (all the original numerical features)
num_cols_in_X = [c for c in numerical_features if c in X.columns]
cat_cols_in_X = [c for c in X.columns if c not in num_cols_in_X]

scaler = StandardScaler()

# Fit on training data only
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols_in_X] = scaler.fit_transform(X_train[num_cols_in_X])
X_test_scaled[num_cols_in_X] = scaler.transform(X_test[num_cols_in_X])

print(f'✅ Scaled {len(num_cols_in_X)} numerical features.')
print(f'   Kept {len(cat_cols_in_X)} encoded categorical features unscaled (binary).')
print(f'\nScaled training data sample:')
X_train_scaled.head()

---
## 5. Exploratory Data Analysis (EDA)

### 5.1 Active vs. Not Active Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
labels = {1: 'Active (1)', 0: 'Not Active (0)'}
counts = df['is_active'].value_counts().sort_index()
colors = ['#e74c3c', '#2ecc71']

axes[0].bar([labels[i] for i in counts.index], counts.values, color=colors)
axes[0].set_title('Customer Count by Activity Status', fontsize=14)
axes[0].set_ylabel('Number of Customers')
for i, (idx, val) in enumerate(counts.items()):
    axes[0].text(i, val + 100, f'{val:,}', ha='center', fontweight='bold')

# Pie chart
axes[1].pie(
    counts.values,
    labels=[labels[i] for i in counts.index],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=[0.05, 0.05]
)
axes[1].set_title('Activity Status Proportion', fontsize=14)

plt.tight_layout()
plt.savefig('../reports/eda_target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'📊 Class Balance Ratio: {counts.min()/counts.max():.2f}')

### 5.2 Feature Distributions

In [ ]:
# ── Distribution of numerical features by target ──
plot_features = [
    'recency', 'total_orders', 'total_revenue', 'customer_lifetime_days',
    'avg_order_value', 'purchase_frequency', 'avg_days_between_orders', 'age'
]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.ravel()

for i, col in enumerate(plot_features):
    for label, color, name in [(0, '#e74c3c', 'Not Active'), (1, '#2ecc71', 'Active')]:
        subset = df[df['is_active'] == label][col]
        axes[i].hist(subset, bins=30, alpha=0.5, label=name, color=color, density=True)
    axes[i].set_title(col, fontsize=12)
    axes[i].legend(fontsize=9)

plt.suptitle('Feature Distributions by Activity Status', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Correlation Analysis

In [ ]:
# ── Correlation heatmap ──
corr_cols = numerical_features + ['is_active']
corr_matrix = df[corr_cols].corr()

plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5
)
plt.title('Feature Correlation Matrix', fontsize=16)
plt.tight_layout()
plt.savefig('../reports/eda_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Correlation with target ──
print('\n📊 Correlation with is_active (sorted by absolute value):')
target_corr = corr_matrix['is_active'].drop('is_active').sort_values(key=abs, ascending=False)
for feat, val in target_corr.items():
    direction = '↑' if val > 0 else '↓'
    print(f'  {feat:30s}  {val:+.4f}  {direction}')

### 5.4 Key Relationships: Recency & Activity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Recency boxplot by target
sns.boxplot(data=df, x='is_active', y='recency', ax=axes[0],
            palette=['#e74c3c', '#2ecc71'])
axes[0].set_xticklabels(['Not Active (0)', 'Active (1)'])
axes[0].set_title('Recency by Activity Status')
axes[0].set_ylabel('Days Since Last Order')

# 2. Total orders boxplot by target
sns.boxplot(data=df, x='is_active', y='total_orders', ax=axes[1],
            palette=['#e74c3c', '#2ecc71'])
axes[1].set_xticklabels(['Not Active (0)', 'Active (1)'])
axes[1].set_title('Total Orders by Activity Status')
axes[1].set_ylabel('Total Orders')

# 3. Customer lifetime boxplot by target
sns.boxplot(data=df, x='is_active', y='customer_lifetime_days', ax=axes[2],
            palette=['#e74c3c', '#2ecc71'])
axes[2].set_xticklabels(['Not Active (0)', 'Active (1)'])
axes[2].set_title('Customer Lifetime by Activity Status')
axes[2].set_ylabel('Lifetime (Days)')

plt.suptitle('Customer Behavior vs Activity Status', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig('../reports/eda_behavior_vs_activity.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Mean feature values by activity status ──
comparison = df.groupby('is_active')[numerical_features].mean().T
comparison.columns = ['Not Active (0)', 'Active (1)']
comparison['Difference'] = comparison['Active (1)'] - comparison['Not Active (0)']
comparison['% Diff'] = (comparison['Difference'] / comparison['Not Active (0)'] * 100).round(1)
comparison = comparison.round(2)
print('📊 Mean Feature Values by Activity Status:')
display(comparison)

---
## 6. Machine Learning — Model Training & Comparison

We train **four models** and compare them:
1. Logistic Regression
2. Decision Tree
3. Random Forest
4. XGBoost

In [ ]:
# ── Define models ──
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, class_weight='balanced'
    ),
    'Decision Tree': DecisionTreeClassifier(
        max_depth=10, min_samples_split=10, random_state=42, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=15, min_samples_split=10,
        random_state=42, class_weight='balanced', n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss',
        n_jobs=-1,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()   # handle class imbalance
    )
}

print(f'📋 Models to train: {list(models.keys())}')

In [ ]:
# ── Train all models and collect predictions ──
results = {}

for name, model in models.items():
    print(f'\n🔧 Training: {name}...')
    
    # Train
    model.fit(X_train_scaled, y_train)
    
    # Predict
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Store results
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'pr_auc': average_precision_score(y_test, y_proba),
        # Metrics specifically for class 0 (Not Active)
        'precision_0': precision_score(y_test, y_pred, pos_label=0, zero_division=0),
        'recall_0': recall_score(y_test, y_pred, pos_label=0),
        'f1_0': f1_score(y_test, y_pred, pos_label=0),
    }
    
    print(f'   ✅ Accuracy: {results[name]["accuracy"]:.4f} | '
          f'F1: {results[name]["f1"]:.4f} | '
          f'ROC-AUC: {results[name]["roc_auc"]:.4f}')

print('\n🎉 All models trained successfully!')

---
## 7. Model Evaluation

### 7.1 Comparison Summary

In [ ]:
# ── Build comparison table ──
comparison_metrics = []
for name, res in results.items():
    comparison_metrics.append({
        'Model': name,
        'Accuracy': res['accuracy'],
        'Precision (1)': res['precision'],
        'Recall (1)': res['recall'],
        'F1-Score (1)': res['f1'],
        'Precision (0)': res['precision_0'],
        'Recall (0)': res['recall_0'],
        'F1-Score (0)': res['f1_0'],
        'ROC-AUC': res['roc_auc'],
        'PR-AUC': res['pr_auc'],
    })

comparison_df = pd.DataFrame(comparison_metrics).set_index('Model').round(4)
print('📊 Model Comparison:')
display(comparison_df)

# Highlight best model by ROC-AUC
best_model_name = comparison_df['ROC-AUC'].idxmax()
print(f'\n🏆 Best Model (by ROC-AUC): {best_model_name} '
      f'(ROC-AUC = {comparison_df.loc[best_model_name, "ROC-AUC"]:.4f})')

In [ ]:
# ── Visual comparison ──
metrics_to_plot = ['Accuracy', 'F1-Score (1)', 'F1-Score (0)', 'ROC-AUC', 'PR-AUC']
plot_df = comparison_df[metrics_to_plot]

ax = plot_df.plot(kind='bar', figsize=(14, 6), rot=0, width=0.75)
ax.set_ylim(0, 1.05)
ax.set_title('Model Performance Comparison', fontsize=16)
ax.set_ylabel('Score')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')

# Add value labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=7, padding=2)

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.2 Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(22, 5))

for ax, (name, res) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Not Active (0)', 'Active (1)'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    ax.set_title(f'{name}\nAcc={res["accuracy"]:.3f}', fontsize=11)

plt.suptitle('Confusion Matrices', fontsize=16, y=1.05)
plt.tight_layout()
plt.savefig('../reports/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.3 Classification Reports (Focus on Class 0 — Not Active)

In [ ]:
for name, res in results.items():
    print(f'\n{"="*60}')
    print(f'📋 Classification Report: {name}')
    print('='*60)
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Not Active (0)', 'Active (1)'],
        digits=4
    ))

### 7.4 ROC Curves

In [ ]:
plt.figure(figsize=(10, 8))
colors_roc = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12']

for (name, res), color in zip(results.items(), colors_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    auc_val = res['roc_auc']
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_val:.4f})', color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC = 0.5)')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Models', fontsize=16)
plt.legend(fontsize=11, loc='lower right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.5 Precision-Recall Curves

In [ ]:
plt.figure(figsize=(10, 8))

for (name, res), color in zip(results.items(), colors_roc):
    precision_vals, recall_vals, _ = precision_recall_curve(y_test, res['y_proba'])
    pr_auc_val = res['pr_auc']
    plt.plot(recall_vals, precision_vals,
             label=f'{name} (PR-AUC = {pr_auc_val:.4f})', color=color, linewidth=2)

# Baseline: proportion of positives
baseline = y_test.mean()
plt.axhline(y=baseline, color='k', linestyle='--', alpha=0.4,
            label=f'Baseline ({baseline:.2f})')

plt.xlabel('Recall', fontsize=12)
plt.ylabel('Precision', fontsize=12)
plt.title('Precision-Recall Curves — All Models', fontsize=16)
plt.legend(fontsize=11, loc='upper right')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()

### 7.6 Feature Importance (Best Model)

In [ ]:
# ── Feature importance from the best model ──
best_res = results[best_model_name]
best_model = best_res['model']

# Get feature importances (works for tree-based models)
if hasattr(best_model, 'feature_importances_'):
    importance = best_model.feature_importances_
elif hasattr(best_model, 'coef_'):
    importance = np.abs(best_model.coef_[0])
else:
    importance = None

if importance is not None:
    feat_imp = pd.DataFrame({
        'Feature': X.columns,
        'Importance': importance
    }).sort_values('Importance', ascending=True)

    plt.figure(figsize=(10, 8))
    plt.barh(feat_imp['Feature'], feat_imp['Importance'], color='#3498db')
    plt.xlabel('Importance', fontsize=12)
    plt.title(f'Feature Importance — {best_model_name}', fontsize=16)
    plt.tight_layout()
    plt.savefig('../reports/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f'\n📊 Top 5 Most Important Features ({best_model_name}):')
    for _, row in feat_imp.tail(5).iloc[::-1].iterrows():
        print(f'  {row["Feature"]:30s}  {row["Importance"]:.4f}')

### 7.7 Focus: Ability to Identify NOT ACTIVE Customers (Class 0)

In [ ]:
# ── Summary table focusing on Class 0 (Not Active) detection ──
class0_summary = []
for name, res in results.items():
    cm = confusion_matrix(y_test, res['y_pred'])
    TN, FP, FN, TP = cm.ravel()
    class0_summary.append({
        'Model': name,
        'True Negatives (correctly ID Not Active)': TN,
        'False Positives (missed Not Active)': FP,
        'Precision (0)': res['precision_0'],
        'Recall (0) — Detection Rate': res['recall_0'],
        'F1 (0)': res['f1_0'],
    })

class0_df = pd.DataFrame(class0_summary).set_index('Model').round(4)
print('📊 NOT ACTIVE (Class 0) Detection Performance:')
print('   Higher Recall(0) = better at catching customers who will churn.\n')
display(class0_df)

best_recall0 = class0_df['Recall (0) — Detection Rate'].idxmax()
print(f'\n🏆 Best at detecting NOT ACTIVE customers: {best_recall0} '
      f'(Recall = {class0_df.loc[best_recall0, "Recall (0) — Detection Rate"]:.4f})')

### 7.8 Cross-Validation (is the model ranking real?)

A single 80/20 split can make differences of ~0.01 AUC look meaningful. We run **5-fold stratified CV on the training set** and keep the out-of-fold probabilities for threshold tuning in the next section.


In [ ]:
# ── 5-fold stratified CV on the training set ──
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof = {}          # out-of-fold probabilities per model
cv_rows = []

for name, model in models.items():
    oof_proba = np.zeros(len(X_train_scaled))
    aucs, praucs = [], []
    for tr_idx, va_idx in skf.split(X_train_scaled, y_train):
        m = clone(model)
        m.fit(X_train_scaled.iloc[tr_idx], y_train.iloc[tr_idx])
        p = m.predict_proba(X_train_scaled.iloc[va_idx])[:, 1]
        oof_proba[va_idx] = p
        aucs.append(roc_auc_score(y_train.iloc[va_idx], p))
        praucs.append(average_precision_score(y_train.iloc[va_idx], p))
    oof[name] = oof_proba
    cv_rows.append({
        'Model': name,
        'CV ROC-AUC (mean)': np.mean(aucs), 'CV ROC-AUC (std)': np.std(aucs),
        'CV PR-AUC (mean)': np.mean(praucs), 'CV PR-AUC (std)': np.std(praucs),
    })
    print(f'✅ {name:22s} CV ROC-AUC = {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')

cv_df = pd.DataFrame(cv_rows).set_index('Model').round(4)
display(cv_df)


### 7.9 Decision-Threshold Tuning

Models output probabilities, and the default cut-off of 0.5 is rarely optimal for an imbalanced target (e.g. XGBoost predicting almost everyone as Not Active). We choose, per model, the threshold that maximises **macro-F1** on the **out-of-fold training predictions** (so the test set is never used for tuning), then evaluate on the test set.


In [ ]:
# ── Threshold tuning on out-of-fold predictions, evaluation on the test set ──
from sklearn.metrics import balanced_accuracy_score

thresholds = np.linspace(0.05, 0.95, 91)
best_thresholds = {}
tuned_rows = []

for name, res in results.items():
    f1s = [f1_score(y_train, (oof[name] >= t).astype(int), average='macro') for t in thresholds]
    t_best = float(thresholds[int(np.argmax(f1s))])
    best_thresholds[name] = t_best

    for label, t in [('default 0.50', 0.5), (f'tuned {t_best:.2f}', t_best)]:
        pred = (res['y_proba'] >= t).astype(int)
        tuned_rows.append({
            'Model': name, 'Threshold': label,
            'Accuracy': accuracy_score(y_test, pred),
            'Balanced Acc': balanced_accuracy_score(y_test, pred),
            'Precision (1)': precision_score(y_test, pred, zero_division=0),
            'Recall (1)': recall_score(y_test, pred),
            'Precision (0)': precision_score(y_test, pred, pos_label=0, zero_division=0),
            'Recall (0)': recall_score(y_test, pred, pos_label=0),
            'Macro-F1': f1_score(y_test, pred, average='macro'),
        })

tuned_df = pd.DataFrame(tuned_rows).set_index(['Model']).round(4)
print('📊 Default vs tuned threshold (test set):')
display(tuned_df)


### 7.10 Baseline Comparison

A model only adds value if it beats a trivial classifier that always predicts the majority class.


In [ ]:
# ── Majority-class baseline ──
baseline_acc = max(y_test.mean(), 1 - y_test.mean())
print(f'Majority-class baseline → accuracy: {baseline_acc:.4f} | balanced accuracy: 0.5000 | ROC-AUC: 0.5000')

best_tuned = tuned_df[tuned_df['Threshold'] != 'default 0.50']['Balanced Acc'].idxmax()
print(f'Best tuned model by balanced accuracy: {best_tuned} '
      f'({tuned_df[(tuned_df.index == best_tuned) & (tuned_df["Threshold"] != "default 0.50")]["Balanced Acc"].iloc[0]:.4f} vs 0.5000 baseline)')


---
## 8. Final Summary & Conclusions

In [ ]:
print('=' * 70)
print('           CUSTOMER CHURN PREDICTION — FINAL SUMMARY')
print('=' * 70)
print(f'\n📦 Dataset: {len(df):,} customers from Gold Layer')
print(f'📊 Features: {X.shape[1]} ({len(num_cols_in_X)} numerical + {len(cat_cols_in_X)} encoded categorical)')
print(f'🎯 Target: is_active  (1=Active, 0=Not Active) | active share = {y.mean():.2%}')
print(f'📅 Reference Date: {REFERENCE_DATE} | feature cutoff: {FEATURE_CUTOFF}')
print(f'🔀 Split: 80% train / 20% test (stratified) + 5-fold CV on train')
print(f'\n📊 Models Trained: {len(models)}')

print(f'\n{"─" * 70}')
print('MODEL RANKINGS (by 5-fold CV ROC-AUC, mean ± std):')
print('─' * 70)
rankings = cv_df.sort_values('CV ROC-AUC (mean)', ascending=False)
for i, (name, row) in enumerate(rankings.iterrows(), 1):
    medal = ['🥇', '🥈', '🥉', '4️⃣'][i-1]
    print(f'{medal} {name:22s} │ CV AUC: {row["CV ROC-AUC (mean)"]:.4f} ± {row["CV ROC-AUC (std)"]:.4f} '
          f'│ Test AUC: {results[name]["roc_auc"]:.4f}')

cv_best = rankings.index[0]
gap = rankings['CV ROC-AUC (mean)'].iloc[0] - rankings['CV ROC-AUC (mean)'].iloc[1]
print(f'\n🏆 Best by CV ROC-AUC: {cv_best}')
if gap < rankings['CV ROC-AUC (std)'].iloc[0]:
    print('   ⚠️ Gap to runner-up is smaller than one CV std → models are statistically similar.')

print(f'\n{"─" * 70}')
print('AFTER THRESHOLD TUNING (test set, threshold chosen on out-of-fold train predictions):')
print('─' * 70)
display(tuned_df[tuned_df['Threshold'] != 'default 0.50'][['Accuracy', 'Balanced Acc', 'Recall (0)', 'Recall (1)', 'Macro-F1']])
print(f'\nMajority-class baseline accuracy: {baseline_acc:.4f} | balanced accuracy: 0.5000 | ROC-AUC: 0.5000')
print('\n✅ All reports saved to: ../reports/')
